# Iteration 2, corrected notebook

This file does not replace `notebooks/colab_iteration2.ipynb`. The older notebook keeps the T4 outputs already recorded.

Set `REVISION` to the 40-character commit that contains the audit fixes. The notebook checks out that commit and will not follow a newer `main` by itself. If this commit is not on GitHub yet, Colab cannot run it: push the commit first, paste its hash, then open this notebook again.

Run Cell 1 after every disconnect, then one experiment cell. Do not start a second experiment while one is running.

In [3]:
# Cell 1 — clone the recorded revision. Run this again after a disconnect.
import os
import re
import subprocess
from pathlib import Path

REVISION = "d143dbdbd690f70315cb706403980eddf09279e9"  # paste the 40-character audit-fix commit; do not leave this blank
REPO = "https://github.com/nazndev/visibility-gated-clip-reid.git"
ROOT = Path("/content/lvlm-person-reid")

if not re.fullmatch(r"[0-9a-f]{40}", REVISION):
    raise SystemExit("Set REVISION to the pushed audit-fix commit. This notebook does not track origin/main.")
if not (ROOT / ".git").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
subprocess.check_call(["git", "fetch", "origin"], cwd=ROOT)
subprocess.check_call(["git", "checkout", "--force", REVISION], cwd=ROOT)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != REVISION:
    raise SystemExit(f"HEAD {head} is not the recorded revision {REVISION}")
os.chdir(ROOT)
print(head)


d143dbdbd690f70315cb706403980eddf09279e9


In [4]:
# Cell 2 — GPU check.
import os
print("nvidia device", os.path.exists("/dev/nvidia0"))
if not os.path.exists("/dev/nvidia0"):
    raise RuntimeError("Select a GPU runtime, then run this cell again.")


nvidia device True


In [5]:
# Cell 3 — install.
import subprocess
from pathlib import Path
ROOT = Path("/content/lvlm-person-reid")
subprocess.check_call(["pip", "install", "-q", "-r", "requirements-colab.txt"], cwd=ROOT)
subprocess.check_call(["pip", "install", "-q", "-e", "."], cwd=ROOT)
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")


2.15.0+cu132 True Tesla T4


In [6]:
# Cell 4 — Drive, dataset, and checkpoint.
import subprocess
from pathlib import Path
from google.colab import drive
ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
drive.mount("/content/drive")
DRIVE.mkdir(parents=True, exist_ok=True)
if not (ROOT / "data/Market-1501-v15.09.15/bounding_box_train").is_dir():
    subprocess.check_call(["python", "scripts/01_download_dataset.py"], cwd=ROOT)
if not (ROOT / "checkpoints/Market1501_clipreid_ViT-B-16_60.pth").is_file():
    subprocess.check_call(["python", "scripts/02_download_model.py"], cwd=ROOT)
print(DRIVE)


Mounted at /content/drive
/content/drive/MyDrive/visibility-gated-reid/iteration2_v2


In [ ]:
# Cell 5 — one experiment. Change only this cell, then run it alone.
# Pending on the same T4, in this order:
#   validation / none / keep 128
#   validation / cls_attention / block 6 / 64
#   validation / cls_attention / block 9 / 96
#   validation / random / block 9 / 96 / seeds 11, 23, 37, 53, 71
# The old block-6 random files stay in iteration2/. This cell writes iteration2_v2/.
SPLIT = "validation"
STRATEGY = "none"  # none, cls_attention, random
BLOCK = "6"
KEEP = "128"
SEED = ""  # 11, 23, 37, 53, or 71 when STRATEGY is random
BACKEND = "reference"

import json
import subprocess
from pathlib import Path
ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
if STRATEGY == "random" and SEED not in {"11", "23", "37", "53", "71"}:
    raise SystemExit("Random selection needs SEED 11, 23, 37, 53, or 71.")
name = f"{SPLIT}_{STRATEGY}_b{BLOCK}_k{KEEP}_{BACKEND}" + (f"_s{SEED}" if STRATEGY == "random" else "")
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", SPLIT,
    "--strategy", STRATEGY,
    "--prune-block", BLOCK,
    "--keep", KEEP,
    "--backend", BACKEND,
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
if STRATEGY == "random":
    command.extend(["--seed", SEED])
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


In [7]:
# Cell 5b — validation, class attention, block 6, keep 64. Run this cell alone.
# Cell 5 already finished the full encoder. Do not run Cell 5 again.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "validation_cls_attention_b6_k64_reference"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "validation",
    "--strategy", "cls_attention",
    "--prune-block", "6",
    "--keep", "64",
    "--backend", "reference",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split validation --strategy cls_attention --prune-block 6 --keep 64 --backend reference --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_cls_attention_b6_k64_reference --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_cls_attention_b6_k64_reference.json
{
  "evaluation": {
    "protocol": "training-derived validation",
    "identity_seen": true,
    "note": "The encoder was trained on these identities and source training images. This is not an official Market-1501 test result."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "1c2cd1835fffe4ff59ca81cf7cd6b69e0abe828df799984612e015d6ad02aa90",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version"

In [ ]:
# Cell 6 — list completed, missing, and failed score files. This does not start a run.
import json
from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
expected = [
    "validation_none_b6_k128_reference.json",
    "validation_cls_attention_b6_k64_reference.json",
    "validation_cls_attention_b9_k96_reference.json",
] + [f"validation_random_b9_k96_reference_s{seed}.json" for seed in (11, 23, 37, 53, 71)]
expected += [
    "test_none_b6_k128_reference.json",
    "test_cls_attention_b6_k64_reference.json",
    "test_cls_attention_b9_k96_reference.json",
]
expected += [f"test_random_b6_k64_reference_s{seed}.json" for seed in (11, 23, 37, 53, 71)]
expected += [f"test_random_b9_k96_reference_s{seed}.json" for seed in (11, 23, 37, 53, 71)]
rows = []
for name in expected:
    path = DRIVE / name
    if not path.is_file():
        rows.append({"file": name, "state": "pending"})
        continue
    payload = json.loads(path.read_text())
    scores = payload.get("scores", {})
    rows.append({
        "file": name,
        "state": payload.get("label"),
        "mAP": scores.get("mAP"),
        "rank1": scores.get("rank1"),
        "rank5": scores.get("rank5"),
        "rank10": scores.get("rank10"),
        "valid_queries": scores.get("valid_queries"),
        "excluded_queries": scores.get("excluded_queries"),
        "kreciprocal": payload.get("kreciprocal"),
        "execution_sha256": payload.get("execution_sha256"),
        "git": payload.get("git"),
        "counts": payload.get("counts"),
    })
print(json.dumps(rows, indent=2))


In [8]:
# Cell 5c — validation, class attention, block 9, keep 96. Run this cell alone.
# Cells 5 and 5b are already finished. Do not run them again.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "validation_cls_attention_b9_k96_reference"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "validation",
    "--strategy", "cls_attention",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split validation --strategy cls_attention --prune-block 9 --keep 96 --backend reference --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_cls_attention_b9_k96_reference --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_cls_attention_b9_k96_reference.json
{
  "evaluation": {
    "protocol": "training-derived validation",
    "identity_seen": true,
    "note": "The encoder was trained on these identities and source training images. This is not an official Market-1501 test result."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "17a5f47ba42af5567ddf4e4b17666ffccd92d4a5a34c0a9a86463ecd52eaadd4",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version"

In [9]:
# Cell 5d — validation, random, block 9, keep 96, seed 11. Run this cell alone.
# Cells 5, 5b, and 5c are already finished. Do not run them again.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "validation_random_b9_k96_reference_s11"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "validation",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "11",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split validation --strategy random --prune-block 9 --keep 96 --backend reference --seed 11 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_random_b9_k96_reference_s11 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_random_b9_k96_reference_s11.json
{
  "evaluation": {
    "protocol": "training-derived validation",
    "identity_seen": true,
    "note": "The encoder was trained on these identities and source training images. This is not an official Market-1501 test result."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "8dbc8b2a62d537d07f0eb3a27f982271b6f67272a44275e007a222b309dcd41b",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "

In [10]:
# Cell 5e — validation, random, block 9, keep 96, seed 23. Run this cell alone.
# Cells 5, 5b, 5c, and 5d are already finished. Do not run them again.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "validation_random_b9_k96_reference_s23"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "validation",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "23",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split validation --strategy random --prune-block 9 --keep 96 --backend reference --seed 23 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_random_b9_k96_reference_s23 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_random_b9_k96_reference_s23.json
{
  "evaluation": {
    "protocol": "training-derived validation",
    "identity_seen": true,
    "note": "The encoder was trained on these identities and source training images. This is not an official Market-1501 test result."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "ac82a571c68ce4ec21c16605bbdec6a2edb1160b0cd8b6e1b2c8c6301e6447fb",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "

In [ ]:
# Cell 5f — validation, random, block 9, keep 96, seed 37. Run this cell alone.
# Cells 5 through 5e are already finished. Do not run them again, and do not start the next seed until this cell prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "validation_random_b9_k96_reference_s37"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "validation",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "37",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


In [ ]:
# Cell 5g — validation, random, block 9, keep 96, seed 53. Run this cell alone.
# Cells 5 through 5f are already finished. Do not run them again, and do not start the next seed until this cell prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "validation_random_b9_k96_reference_s53"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "validation",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "53",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


In [ ]:
# Cell 5h — validation, random, block 9, keep 96, seed 71. Run this cell alone.
# Cells 5 through 5g are already finished. Do not run them again, and do not start the next seed until this cell prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "validation_random_b9_k96_reference_s71"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "validation",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "71",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


In [ ]:
# Cell 7 — corrected timing. Run only after the accuracy cells you need are finished.
# Planned protocol: batch 1 and 32, 10 warm-up, 50 measured iterations, 3 trials, FP32.
import subprocess
from pathlib import Path
ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
for batch in ("1", "32"):
    subprocess.check_call([
        "python", "-m", "reidvlm", "prune-benchmark",
        "--config", "configs/market1501.yaml",
        "--batch-size", batch,
        "--warmup", "10",
        "--iterations", "50",
        "--trials", "3",
        "--prune-block", "6",
        "--keep", "64",
        "--seed", "11",
        "--output", str(DRIVE / f"benchmark_b{batch}_corrected.json"),
    ], cwd=ROOT)
